# 01 — Map Industries

Fetches industry/sector for each company in `data/processed/matched_companies.xlsx` via yfinance and writes `data/processed/industry_map.xlsx`, which every downstream notebook joins on.

In [1]:
import pandas as pd
import yfinance as yf
import time
from pathlib import Path

BASE_DIR = Path.cwd()

mc = pd.read_excel(BASE_DIR / "data" / "processed" / "matched_companies.xlsx")
print(f"Loaded {len(mc)} companies")
mc.head()

Loaded 247 companies


,BSE Name,BSE Code,Prowess Name,Prowess Code,NSE Symbol
0,360 ONE WAM LIMITED,542772,360 One Wam Ltd.,384392.0,360ONE
1,3M INDIA LIMITED,523395,3M India Ltd.,36277.0,3MINDIA
2,ABB India Limited,500002,A B B India Ltd.,21420.0,ABB
3,ACC LIMITED,500410,A C C Ltd.,23354.0,ACC
4,ADANI GREEN ENERGY LIMITED,541450,Adani Green Energy Ltd.,527121.0,ADANIGREEN


In [2]:
def fetch_industry(nse_symbol):
    """Fetch industry and sector from yfinance using NSE symbol."""
    try:
        info = yf.Ticker(f"{nse_symbol}.NS").info
        return info.get("industry"), info.get("sector")
    except Exception:
        return None, None

results = []
for _, row in mc.iterrows():
    symbol = row["NSE Symbol"]
    industry, sector = fetch_industry(symbol)
    results.append({
        "BSE Code":    row["BSE Code"],
        "BSE Name":    row["BSE Name"],
        "NSE Symbol":  symbol,
        "Industry":    industry,
        "Sector":      sector,
    })
    print(f"{symbol:20s} → {industry or 'N/A'}")
    time.sleep(0.3)   # avoid rate limiting

industry_df = pd.DataFrame(results)
print(f"\nMapped:   {industry_df['Industry'].notna().sum()} / {len(industry_df)}")
print(f"Unmapped: {industry_df['Industry'].isna().sum()}")

360ONE               → Asset Management
3MINDIA              → Conglomerates
ABB                  → Specialty Industrial Machinery
ACC                  → Building Materials
ADANIGREEN           → Utilities - Renewable
ADANIPOWER           → Utilities - Independent Power Producers
ABSLAMC              → Asset Management
AFFLE                → Advertising Agencies
AIAENG               → Specialty Industrial Machinery
ALKEM                → Drug Manufacturers - Specialty & Generic
AMBER                → Furnishings, Fixtures & Appliances
AMBUJACEM            → Building Materials
ANANTRAJ             → Real Estate - Development
APARINDS             → Electrical Equipment & Parts
APOLLOHOSP           → Medical Care Facilities
ASTRAL               → Building Products & Equipment
ASTRAZEN             → Drug Manufacturers - Specialty & Generic
ATUL                 → Specialty Chemicals
AUBANK               → Banks - Regional
AUROPHARMA           → Drug Manufacturers - Specialty & Generic
DMART

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ISEC.NS"}}}


ISEC                 → N/A
IDFCFIRSTB           → Banks - Regional
IFCI                 → Credit Services
IIFL                 → Credit Services
IRFC                 → Credit Services
IGL                  → Utilities - Regulated Gas
INDUSTOWER           → Telecom Services
NAUKRI               → Internet Content & Information
INFY                 → Information Technology Services
INOXWIND             → Specialty Industrial Machinery
INDIGO               → Airlines
IRB                  → Infrastructure Operations
IRCON                → Engineering & Construction
ITC                  → Tobacco
INDIANB              → Banks - Regional
JBCHEPHARM           → Drug Manufacturers - Specialty & Generic
JINDALSTEL           → Steel
JMFINANCIL           → Capital Markets
JUBLFOOD             → Restaurants
JUBLPHARMA           → Drug Manufacturers - Specialty & Generic
KPRMILL              → Textile Manufacturing
KAJARIACER           → Building Products & Equipment
KPIL                 → Engineerin

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PEL.NS"}}}


PEL                  → N/A
PPLPHARMA            → Drug Manufacturers - Specialty & Generic
PFC                  → Credit Services
PRESTIGE             → Real Estate - Diversified
RADICO               → Beverages - Wineries & Distilleries
REDINGTON            → Information Technology Services
RELIANCE             → Oil & Gas Refining & Marketing
RPOWER               → Utilities - Independent Power Producers
RAINBOW              → Medical Care Facilities
MOTHERSON            → Auto Parts
SBICARD              → Credit Services
SHREECEM             → Building Materials
SHYAMMETL            → Steel
SJVN                 → Utilities - Renewable
SOBHA                → Real Estate - Development
SOLARINDS            → Specialty Chemicals
SONACOMS             → Auto Parts
SRF                  → Conglomerates
SAIL                 → Steel
SUMICHEM             → Agricultural Inputs
SUNDRMFAST           → Auto Parts


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SUVENPHARMA.NS"}}}


SUVENPHARMA          → N/A
SARDAEN              → Steel
SCHAEFFLER           → Auto Parts
SUNPHARMA            → Drug Manufacturers - Specialty & Generic
SYNGENE              → Diagnostics & Research
SYRMA                → Electronic Components
TATAELXSI            → Software - Application
TATASTEEL            → Steel
FEDERALBNK           → Banks - Regional
GESHIP               → Marine Shipping
NIACL                → Insurance - Diversified
PHOENIXLTD           → Real Estate - Diversified
RAMCOCEM             → Building Materials
THERMAX              → Conglomerates
TIMKEN               → Tools & Accessories
TRIDENT              → Textile Manufacturing
TRITURBINE           → Specialty Industrial Machinery
TVSHLTD              → Auto Parts
TATACHEM             → Chemicals
TATACONSUM           → Packaged Foods
TATAINVEST           → Asset Management
TMCV                 → Auto Manufacturers
TECHNOE              → Engineering & Construction
INDHOTEL             → Lodging
SUPREMEIND      

In [ ]:
# Show unmapped companies (if any need manual fixing)
print("Unmapped companies:")
print(industry_df[industry_df["Industry"].isna()][["BSE Code", "BSE Name", "NSE Symbol"]].to_string(index=False))

In [4]:
# Save to file
out_path = BASE_DIR / "data" / "processed" / "industry_map.xlsx"
industry_df.to_excel(out_path, index=False)
industry_df

,BSE Code,BSE Name,NSE Symbol,Industry,Sector
0,542772,360 ONE WAM LIMITED,360ONE,Asset Management,Financial Services
1,523395,3M INDIA LIMITED,3MINDIA,Conglomerates,Industrials
2,500002,ABB India Limited,ABB,Specialty Industrial Machinery,Industrials
3,500410,ACC LIMITED,ACC,Building Materials,Basic Materials
4,541450,ADANI GREEN ENERGY LIMITED,ADANIGREEN,Utilities - Renewable,Utilities
...,...,...,...,...,...
242,532300,Wockhardt Limited,WOCKPHARMA,Drug Manufacturers - Specialty & Generic,Healthcare
243,532648,YES BANK LIMITED,YESBANK,Banks - Regional,Financial Services
244,533023,ZF COMMERCIAL VEHICLE CONTROL SYSTEMS INDIA LI...,ZFCVINDIA,Auto Parts,Consumer Cyclical
245,543320,ZOMATO LIMITED,ZOMATO,NaN,NaN
